## Rational Numbers

In [5]:
from sympy import Rational

display(Rational(1, 2))
display(Rational('1/2'))

1/2

1/2

## Constants & Physical Units

In [9]:
from sympy.physics.units import Quantity, length, meter
from sympy.physics.units.systems.si import SI
from sympy.physics.units import convert_to

Proteus = Quantity('P')

# 'Proteus' is a measurement of length
SI.set_quantity_dimension(Proteus, length)

# conversion factor (1 P= 1.114514 meters)
SI.set_quantity_scale_factor(Proteus, 1.114514 * meter)

display(convert_to(1919810 * meter, Proteus))

1722553.50762754⋅P

In [2]:
from sympy.physics.units import convert_to, speed_of_light, meter, second, liters, hour

from sympy import init_printing
init_printing(fold_short_frac=True) # Use inline format for simple fractions

display(convert_to(speed_of_light, [meter, second]))
display(convert_to(speed_of_light, [meter, hour]))
init_printing(fold_short_frac=False)

299792458⋅meter
───────────────
    second     

1079252848800⋅meter
───────────────────
       hour        

In [3]:
from sympy.physics.units import meters, kilometer
from sympy.physics.units.util import quantity_simplify

display((1000 * meters/ (2 * kilometer)))
display((1000 * meters/ (2 * kilometer)).simplify())

500⋅meter
─────────
kilometer

1/2

## Filter Results in Range for Non-linear Equations

In [4]:
from sympy import symbols, cos, nonlinsolve, Eq, Set, Interval, EmptySet
from IPython.display import display, Markdown
import itertools

def filter(sols, domains, param_name="n"):
    assert isinstance(sols, Set), f"Solution DNE, got {type(sols)}"
    ret = []
    # Iterate outer solution set
    for sol in sols.args:
        print(f"\n[DEBUG] Current solution: {sol}")
        
        # Then iterate varables inside a solution, filter variable value with the domain specified,
        # and enumerate all possible groups of variables that its value is an interval
        filed_vars = []
        for i, var in enumerate(sol):
            print(f"[DEBUG][{i}] type={type(var).__name__}")
            print(f"[DEBUG]    value: {var}")
            
            # If this variable is an interval
            if isinstance(var, Set):
                # filter using intersect
                if (filed_var := var.intersect(domains[i])) != EmptySet:
                    filed_vars.append(filed_var)
            else:  # if just a single number:
                if var in domains[i]: filed_vars.append({var})
        
        # If lacks one variable, drop this solution
        if len(filed_vars) < len(sol):
            print(f"[DEBUG]    dropped filtered solution: {filed_vars}")
            break
        else:
            print(f"[DEBUG]    filtered solution: {filed_vars}")
        
        # Rearrange solutions with cartesian product (e.g. every possible x, y, z combination)
        enumed_sol = []
        enumed_sol += itertools.product(*filed_vars)
        print(f"[DEBUG]    enumerated solution: {enumed_sol}")

        ret.append(set(enumed_sol))  # deduplicates with hashset
    return ret

if __name__ == "__main__":
    x, y, z = symbols("x y z", real=True)

    # System of equations, with no definite solution
    eq1 = Eq(x * y, z)
    eq2 = Eq(-x * y, z)
    eq3 = Eq(cos(x) + sin(y), z)
    
    sols = nonlinsolve([eq1, eq2, eq3], [x, y, z])

    display(Markdown("Raw solutions:"))
    display(sols)

    # We need make sure the solution set is reasonable. restrict the domain for each variable
    domains = [
        Interval(-10, 10),
        Interval(-10, 10),
        Interval(-10, 10),
    ]
    filtered_sols = filter(sols, domains)
    
    display(Markdown("Filtered solutions:"))
    for i in filtered_sols:
        display(i)

Raw solutions:

{(0, ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers), 0), (ImageSet(Lambda(_n, 2*_n*pi + pi/2), Integers), 0, 0), (ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers), 0, 0)}


[DEBUG] Current solution: (0, ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers), 0)
[DEBUG][0] type=Zero
[DEBUG]    value: 0
[DEBUG][1] type=ImageSet
[DEBUG]    value: ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers)
[DEBUG][2] type=Zero
[DEBUG]    value: 0
[DEBUG]    filtered solution: [{0}, {-5*pi/2, -pi/2, 3*pi/2}, {0}]
[DEBUG]    enumerated solution: [(0, -5*pi/2, 0), (0, -pi/2, 0), (0, 3*pi/2, 0)]

[DEBUG] Current solution: (ImageSet(Lambda(_n, 2*_n*pi + pi/2), Integers), 0, 0)
[DEBUG][0] type=ImageSet
[DEBUG]    value: ImageSet(Lambda(_n, 2*_n*pi + pi/2), Integers)
[DEBUG][1] type=Zero
[DEBUG]    value: 0
[DEBUG][2] type=Zero
[DEBUG]    value: 0
[DEBUG]    filtered solution: [{-3*pi/2, pi/2, 5*pi/2}, {0}, {0}]
[DEBUG]    enumerated solution: [(-3*pi/2, 0, 0), (pi/2, 0, 0), (5*pi/2, 0, 0)]

[DEBUG] Current solution: (ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers), 0, 0)
[DEBUG][0] type=ImageSet
[DEBUG]    value: ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers)
[DEBUG][1] type=Ze

Filtered solutions:

{(0, -5*pi/2, 0), (0, -pi/2, 0), (0, 3*pi/2, 0)}

{(-3*pi/2, 0, 0), (pi/2, 0, 0), (5*pi/2, 0, 0)}

{(-5*pi/2, 0, 0), (-pi/2, 0, 0), (3*pi/2, 0, 0)}